This notebook ingests and prepares data from the UK Census 2021 (ONS) ethnicity dataset, providing detailed population breakdowns by ethnic group across geographies. The purpose is to establish accurate population denominators for ethnicity at borough level (including NW London), which are essential for contextualising health outcomes and enabling comparisons across communities.

In [2]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import pandas as pd

HERE = Path.cwd()
PARQUET_ENGINE = "pyarrow" if importlib.util.find_spec("pyarrow") else "fastparquet"

CENSUS_PATH = HERE / "raw_census2021_ethnicity_ts021.csv"
BOROUGH_PATH = HERE / "_config_nwl_boroughs.csv"
ETH_MAP_PATH = HERE / "_config_ethnicity_mapping.csv"

OUT_PATH = HERE / "_processed_census2021_ethnicity_nwl.parquet"
QA_PATH = HERE / "_qa_census_ethnicity_ingestion_issues.csv"



In [3]:
# Load data 
if not CENSUS_PATH.exists():
    raise FileNotFoundError("Census ethnicity CSV not found.")

census = pd.read_csv(CENSUS_PATH)
census.shape

(6620, 5)

In [4]:
# Load NWL borough list
if not BOROUGH_PATH.exists():
    raise FileNotFoundError(f"Missing file: {BOROUGH_PATH.name}")

boroughs = pd.read_csv(BOROUGH_PATH)

# Assumption: first column is borough name list (matches your existing project style)
borough_col = boroughs.columns[0]
nwl_boroughs = boroughs[borough_col].astype(str).str.strip().tolist()

len(nwl_boroughs), nwl_boroughs[:10]

(8,
 ['Brent',
  'Ealing',
  'Hammersmith and Fulham',
  'Harrow',
  'Hillingdon',
  'Hounslow',
  'Kensington and Chelsea',
  'Westminster'])

In [5]:
# Filter to NWL (using exact Census column names)
geo_code_col = "Lower Tier Local Authorities Code"
geo_name_col = "Lower Tier Local Authorities"
eth_col = "Ethnic group (20 categories)"
eth_code_col = "Ethnic group (20 categories) Code"
obs_col = "Observation"

required = {geo_code_col, geo_name_col, eth_col, eth_code_col, obs_col}
missing = required - set(census.columns)
if missing:
    raise KeyError(f"Census missing columns: {sorted(missing)}")

census[obs_col] = pd.to_numeric(census[obs_col], errors="coerce")

census_nwl = census[census[geo_name_col].astype(str).str.strip().isin(nwl_boroughs)].copy()
census_nwl.shape

(160, 5)

In [6]:
# Borough totals + proportions
totals = (
    census_nwl
    .groupby([geo_code_col, geo_name_col], as_index=False)[obs_col]
    .sum()
    .rename(columns={obs_col: "borough_total"})
)

census_nwl = census_nwl.merge(totals, on=[geo_code_col, geo_name_col], how="left")

census_nwl = census_nwl.rename(columns={obs_col: "count"})
census_nwl["proportion"] = census_nwl["count"] / census_nwl["borough_total"]

census_nwl[[geo_name_col, eth_col, "count", "borough_total", "proportion"]].head(10)


,Lower Tier Local Authorities,Ethnic group (20 categories),count,borough_total,proportion
0,Brent,Does not apply,0,339821,0.000000
1,Brent,"Asian, Asian British or Asian Welsh: Bangladeshi",2186,339821,0.006433
2,Brent,"Asian, Asian British or Asian Welsh: Chinese",3393,339821,0.009985
3,Brent,"Asian, Asian British or Asian Welsh: Indian",66157,339821,0.194682
4,Brent,"Asian, Asian British or Asian Welsh: Pakistani",15217,339821,0.044779
5,Brent,"Asian, Asian British or Asian Welsh: Other Asian",24562,339821,0.072279
6,Brent,"Black, Black British, Black Welsh, Caribbean o...",31070,339821,0.091430
7,Brent,"Black, Black British, Black Welsh, Caribbean o...",21258,339821,0.062556
8,Brent,"Black, Black British, Black Welsh, Caribbean o...",7167,339821,0.021091
9,Brent,Mixed or Multiple ethnic groups: White and Asian,3607,339821,0.010614


In [7]:
# Ethnicity standardisation (reuse a mapping file)
if not ETH_MAP_PATH.exists():
    raise FileNotFoundError(f"Missing file: {ETH_MAP_PATH.name}")

eth_map = pd.read_csv(ETH_MAP_PATH)
required_map_cols = {"raw_ethnicity", "standard_ethnicity"}
missing_map = required_map_cols - set(eth_map.columns)
if missing_map:
    raise KeyError(f"Ethnicity mapping missing columns: {sorted(missing_map)}")

ETH_MAP = dict(
    zip(
        eth_map["raw_ethnicity"].astype(str).str.strip(),
        eth_map["standard_ethnicity"].astype(str).str.strip(),
    )
)

census_nwl["raw_ethnicity"] = census_nwl[eth_col].astype(str).str.strip()
census_nwl["standard_ethnicity"] = (
    census_nwl["raw_ethnicity"].map(ETH_MAP).fillna(census_nwl["raw_ethnicity"])
)

In [8]:
out = pd.DataFrame(index=census_nwl.index)

out["source"] = "ONS Census 2021"
out["dataset"] = "Ethnic group (20 categories) by borough (LTLA)"
out["geography_level"] = "Borough"
out["geography_code"] = census_nwl[geo_code_col].astype(str).str.strip()
out["geography_name"] = census_nwl[geo_name_col].astype(str).str.strip()

out["raw_ethnicity_code"] = census_nwl[eth_code_col].astype(str).str.strip()
out["raw_ethnicity"] = census_nwl["raw_ethnicity"]
out["standard_ethnicity"] = census_nwl["standard_ethnicity"]

out["count"] = census_nwl["count"]
out["borough_total"] = census_nwl["borough_total"]
out["proportion"] = census_nwl["proportion"]

out.tail(10)

,source,dataset,geography_level,geography_code,geography_name,raw_ethnicity_code,raw_ethnicity,standard_ethnicity,count,borough_total,proportion
150,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,10,Mixed or Multiple ethnic groups: White and Bla...,Mixed or Multiple ethnic groups: White and Bla...,2089,204236,0.010228
151,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,11,Mixed or Multiple ethnic groups: White and Bla...,Mixed or Multiple ethnic groups: White and Bla...,2061,204236,0.010091
152,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,12,Mixed or Multiple ethnic groups: Other Mixed o...,Mixed or Multiple ethnic groups: Other Mixed o...,5467,204236,0.026768
153,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,13,"White: English, Welsh, Scottish, Northern Iris...","White: English, Welsh, Scottish, Northern Iris...",57162,204236,0.279882
154,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,14,White: Irish,White: Irish,3742,204236,0.018322
155,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,15,White: Gypsy or Irish Traveller,White: Gypsy or Irish Traveller,49,204236,0.000240
156,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,16,White: Roma,White: Roma,1503,204236,0.007359
157,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,17,White: Other White,White: Other White,50276,204236,0.246166
158,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,18,Other ethnic group: Arab,Other ethnic group: Arab,15439,204236,0.075594
159,ONS Census 2021,Ethnic group (20 categories) by borough (LTLA),Borough,E09000033,Westminster,19,Other ethnic group: Any other ethnic group,Other ethnic group: Any other ethnic group,12032,204236,0.058912


In [9]:
# Mapping for Census wording 

ANALYTICAL_MAP = {
    # White
    "White: English, Welsh, Scottish, Northern Irish or British": "White British",
    "White: Irish": "White Other",
    "White: Gypsy or Irish Traveller": "White Other",
    "White: Roma": "White Other",
    "White: Other White": "White Other",

    # Black (correct Census 2021 wording)
    "Black, Black British, Black Welsh, Caribbean or African: African": "Black African",
    "Black, Black British, Black Welsh, Caribbean or African: Caribbean": "Black Caribbean",
    "Black, Black British, Black Welsh, Caribbean or African: Other Black": "Black Other",

    # Asian
    "Asian, Asian British or Asian Welsh: Indian": "Asian",
    "Asian, Asian British or Asian Welsh: Pakistani": "Asian",
    "Asian, Asian British or Asian Welsh: Bangladeshi": "Asian",
    "Asian, Asian British or Asian Welsh: Chinese": "Asian",
    "Asian, Asian British or Asian Welsh: Other Asian": "Asian",

    # Mixed
    "Mixed or Multiple ethnic groups: White and Black Caribbean": "Mixed",
    "Mixed or Multiple ethnic groups: White and Black African": "Mixed",
    "Mixed or Multiple ethnic groups: White and Asian": "Mixed",
    "Mixed or Multiple ethnic groups: Other Mixed or Multiple ethnic groups": "Mixed",

    # Other
    "Other ethnic group: Arab": "Other",
    "Other ethnic group: Any other ethnic group": "Other",
}

In [10]:
# Exclude 'Does not apply'
out = out[out["raw_ethnicity"] != "Does not apply"].copy()

In [11]:
out["analytical_ethnicity"] = out["raw_ethnicity"].map(ANALYTICAL_MAP)

unmapped = out[out["analytical_ethnicity"].isna()]["raw_ethnicity"].unique()
unmapped


<ArrowStringArray>
[]
Length: 0, dtype: str

In [12]:
# Aggregate 
agg = (
    out.groupby(
        ["geography_code", "geography_name", "analytical_ethnicity"],
        as_index=False
    )
    .agg(
        population=("count", "sum"),
        borough_total=("borough_total", "first")
    )
)

agg["proportion"] = agg["population"] / agg["borough_total"]
agg


,geography_code,geography_name,analytical_ethnicity,population,borough_total,proportion
0,E09000005,Brent,Asian,111515,339821,0.328158
1,E09000005,Brent,Black African,31070,339821,0.091430
2,E09000005,Brent,Black Caribbean,21258,339821,0.062556
3,E09000005,Brent,Black Other,7167,339821,0.021091
4,E09000005,Brent,Mixed,17249,339821,0.050759
...,...,...,...,...,...,...
59,E09000033,Westminster,Black Other,1698,204236,0.008314
60,E09000033,Westminster,Mixed,13335,204236,0.065292
61,E09000033,Westminster,Other,27471,204236,0.134506
62,E09000033,Westminster,White British,57162,204236,0.279882


In [13]:
agg.shape

(64, 6)

In [20]:
# Top boroughs by Black African proportion 
agg.loc[agg["analytical_ethnicity"].eq("Black African")] \
   .sort_values("proportion", ascending=False)[["geography_name", "proportion"]] \
   .head(10)

,geography_name,proportion
1,Brent,0.091430
17,Hammersmith and Fulham,0.072304
9,Ealing,0.061501
33,Hillingdon,0.051794
41,Hounslow,0.051686
57,Westminster,0.051171
49,Kensington and Chelsea,0.048433
25,Harrow,0.040520


In [15]:
# Top borough by Black Carribean proportion 
agg.loc[agg["analytical_ethnicity"].eq("Black Caribbean")] \
   .sort_values("proportion", ascending=False)[["geography_name", "proportion"]] \
   .head(10)

,geography_name,proportion
2,Brent,0.062556
18,Hammersmith and Fulham,0.036177
10,Ealing,0.035133
26,Harrow,0.024938
50,Kensington and Chelsea,0.022577
58,Westminster,0.021088
34,Hillingdon,0.018803
42,Hounslow,0.012398


In [16]:
AGG_OUT_PATH = HERE / "_processed_census2021_ethnicity_nwl_analytical.parquet"
agg.to_parquet(AGG_OUT_PATH, index=False, engine=PARQUET_ENGINE)
AGG_OUT_PATH

WindowsPath('c:/Users/OlenaManziuk/OneDrive - Third Sector Together North West London/Documents/nwl_health_inequalities/_processed_census2021_ethnicity_nwl_analytical.parquet')

The Census 2021 ethnicity data has been cleaned, standardised, and structured into a consistent format aligned with the project’s ethnicity framework. Geographic fields have been harmonised and outputs validated to ensure completeness and accuracy. The resulting dataset provides a reliable population baseline to support modelling and comparative analysis in later stages.